<a href="https://colab.research.google.com/github/pavanreddy02/Embedings/blob/main/Fintech_Embeddings_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the sentence-transformers library
!pip install -q sentence-transformers

# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util

In [ ]:
sentences = [
    # Theme 1: Payments & Banking
    "Digital wallets allow users to make contactless payments instantly.",
    "Mobile banking applications have revolutionized how consumers manage checking accounts.",
    "Cross-border payment gateways are reducing transaction fees for global trade.",
    "Peer-to-peer payment apps make splitting dinner bills effortless.",
    "Central banks are exploring digital currencies to modernize retail payment systems.",

    # Theme 2: Cryptocurrency & DeFi
    "Bitcoin relies on a decentralized ledger to secure peer-to-peer transactions.",
    "Ethereum enables developers to deploy smart contracts without intermediaries.",
    "Decentralized finance protocols allow users to yield farm and lend crypto assets.",
    "Automated market makers provide liquidity to decentralized exchanges.",
    "Non-fungible tokens represent unique ownership of digital assets on a blockchain.",

    # Theme 3: Lending & Credit
    "Peer-to-peer lending platforms connect individual borrowers directly with investors.",
    "Buy now pay later services offer interest-free installments at retail checkouts.",
    "AI-driven credit scoring algorithms evaluate alternative data for loan approvals.",
    "Microfinance institutions leverage fintech to provide loans to unbanked populations.",
    "Automated underwriting speeds up the mortgage approval process significantly.",

    # Theme 4: Insurtech (Insurance Tech)
    "Insurtech startups use telematics data to price auto insurance dynamically.",
    "Parametric insurance pays out automatically when predefined environmental triggers occur.",
    "Digital claims processing uses AI to analyze photos of car accident damage.",
    "Usage-based insurance policies charge premiums based on actual miles driven.",
    "Health insurtech platforms offer discounts to users who track their daily steps."
]

In [ ]:
# Load the pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute the embeddings (this converts text into 384-dimensional vectors)
embeddings = model.encode(sentences, convert_to_tensor=True)

print(f"Embeddings shape: {embeddings.shape}")
# You will see torch.Size([20, 384]) -> 20 sentences, each represented by 384 numbers.

In [ ]:
# Compute the cosine similarity matrix between all 20 sentences
cosine_scores = util.cos_sim(embeddings, embeddings)

# Convert the tensor results to a numpy array for easy plotting
similarity_matrix = cosine_scores.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 10))

# Create a heatmap using seaborn
sns.heatmap(
    similarity_matrix,
    cmap='coolwarm',       # Red = High similarity, Blue = Low similarity
    xticklabels=[f"S{i+1}" for i in range(20)],
    yticklabels=[f"S{i+1}" for i in range(20)],
    annot=False,            # Set to True if you want to see the exact numbers, but it gets crowded
    vmin=0, vmax=1
)

plt.title("Fintech Sentences Cosine Similarity Heatmap", fontsize=16)
plt.xlabel("Sentence Index")
plt.ylabel("Sentence Index")
plt.show()

In [ ]:
# Step 1: Define a new search query
user_query = "How can I trade digital assets without a central company?"

# Step 2: Convert the query into the exact same 384-dimensional vector space
query_embedding = model.encode(user_query, convert_to_tensor=True)

# Step 3: Compute similarity between the single query and all 20 sentences
# This returns an array of 20 scores
search_scores = util.cos_sim(query_embedding, embeddings)[0].cpu().numpy()

# Step 4: Pair each sentence with its score, and sort them from highest to lowest
results = []
for index, score in enumerate(search_scores):
    results.append((score, sentences[index]))

# Sort by the score (element 0 of the tuple) in descending order
results.sort(key=lambda x: x[0], reverse=True)

# Step 5: Print out the top 3 closest matches
print(f"User Query: '{user_query}'\n")
print("Top 3 Semantic Search Matches:")
print("-" * 50)
for rank, (score, sentence) in enumerate(results[:3], 1):
    print(f"Rank {rank} (Similarity Score: {score:.4f})")
    print(f"👉 {sentence}\n")

In [ ]:
!pip install opentelemetry-api==1.38.0 opentelemetry-sdk==1.38.0 opentelemetry-exporter-otlp-proto-common==1.38.0 opentelemetry-proto==1.38.0 opentelemetry-exporter-otlp-proto-grpc==1.38.0 opentelemetry-semantic-conventions==0.59b0 protobuf==5.29.1 --force-reinstall
!pip install -q chromadb

import chromadb
from chromadb.utils import embedding_functions

In [ ]:
# Expanded dataset: 50 structured items
raw_data = [
    {"text": "Instant payment rails process merchant transactions in real-time.", "domain": "Payments", "risk": "low"},
    {"text": "Cross-border clearing networks reduce settlement lag across continents.", "domain": "Payments", "risk": "medium"},
    {"text": "Mobile point-of-sale systems enable contactless hardware integrations.", "domain": "Payments", "risk": "low"},
    {"text": "Automated clearing houses manage high-volume recurring batch deposits.", "domain": "Payments", "risk": "low"},
    {"text": "Interoperable QR code systems standardize retail peer-to-peer transfers.", "domain": "Payments", "risk": "low"},
    {"text": "Digital wallet provision architectures secure card tokenization layers.", "domain": "Payments", "risk": "low"},
    {"text": "Open banking APIs expose core accounting ledgers to third-party providers.", "domain": "Payments", "risk": "high"},
    {"text": "Real-time gross settlement systems execute high-value liquidity moves.", "domain": "Payments", "risk": "low"},
    {"text": "Merchant discount rates optimize transaction revenue distributions dynamically.", "domain": "Payments", "risk": "low"},
    {"text": "Biometric transaction authentication mitigates card-not-present fraud vectors.", "domain": "Payments", "risk": "low"},

    {"text": "Decentralized consensus engines validate blocks without centralized state controls.", "domain": "Crypto", "risk": "high"},
    {"text": "Smart contract conditions evaluate programmatic multi-signature fund releases.", "domain": "Crypto", "risk": "high"},
    {"text": "Zero-knowledge proof mechanics mask transaction payloads on-chain safely.", "domain": "Crypto", "risk": "medium"},
    {"text": "Layer-2 scaling rollup solutions batch computational proofs statefully.", "domain": "Crypto", "risk": "medium"},
    {"text": "Liquidity pool invariant formulas balance decentralized token exchange values.", "domain": "Crypto", "risk": "high"},
    {"text": "Proof-of-stake validation protocols reward nodes for lockup commitments.", "domain": "Crypto", "risk": "low"},
    {"text": "Crypto asset custody solutions leverage multi-party computation algorithms.", "domain": "Crypto", "risk": "medium"},
    {"text": "Cross-chain bridge contracts lock and mint collateral tokens synthetically.", "domain": "Crypto", "risk": "high"},
    {"text": "Stablecoin algorithmic pegs rebalance supply metrics based on market deviations.", "domain": "Crypto", "risk": "high"},
    {"text": "Governance token voting frameworks drive decentralized autonomous protocols.", "domain": "Crypto", "risk": "low"},

    {"text": "Automated underwriting engines compute debt-to-income ratios instantaneously.", "domain": "Lending", "risk": "medium"},
    {"text": "Alternative credit tracking structures analyze rental utilities payment compliance.", "domain": "Lending", "risk": "low"},
    {"text": "Securitized debt portfolios aggregate mortgage originations for secondary markets.", "domain": "Lending", "risk": "medium"},
    {"text": "Micro-lending platform microservices disburse capital allocations to mobile endpoints.", "domain": "Lending", "risk": "medium"},
    {"text": "Dynamic interest rate adjusters parse macro inflation points periodically.", "domain": "Lending", "risk": "low"},
    {"text": "Collateral valuation engines track property equity variations automatically.", "domain": "Lending", "risk": "low"},
    {"text": "Loan servicing applications manage amortization schedules and late fee rules.", "domain": "Lending", "risk": "low"},
    {"text": "Peer-to-peer fractional credit networks pool user investments cleanly.", "domain": "Lending", "risk": "high"},
    {"text": "Commercial lines of credit utilize real-time corporate cash accounts tracking.", "domain": "Lending", "risk": "medium"},
    {"text": "Bridge loan refinancing structures optimize commercial property transition phases.", "domain": "Lending", "risk": "medium"},

    {"text": "Anti-money laundering software flags structurally anomalous transaction flows.", "domain": "Compliance", "risk": "high"},
    {"text": "Know-your-customer checking algorithms parse government database credential arrays.", "domain": "Compliance", "risk": "low"},
    {"text": "Suspicious activity report pipelines automate regulatory oversight compliance filings.", "domain": "Compliance", "risk": "medium"},
    {"text": "Sanctions screening databases block transfers involving prohibited global entities.", "domain": "Compliance", "risk": "high"},
    {"text": "GDPR compliance storage services encrypt personal data identifiers completely.", "domain": "Compliance", "risk": "low"},
    {"text": "Audit trail logging microservices preserve historical system mutation records.", "domain": "Compliance", "risk": "low"},
    {"text": "Regulatory sandboxes isolate trial testing of novel consumer financial systems.", "domain": "Compliance", "risk": "medium"},
    {"text": "Insider trading monitoring engines monitor anomalous corporate stock options timing.", "domain": "Compliance", "risk": "high"},
    {"text": "Tax compliance modules calculate cross-jurisdictional sales levy metrics instantly.", "domain": "Compliance", "risk": "low"},
    {"text": "Fair lending compliance models monitor credit rejection rates for structural bias.", "domain": "Compliance", "risk": "medium"},

    {"text": "Intrusion detection agents identify coordinated hardware credential brute-forcing.", "domain": "Security", "risk": "high"},
    {"text": "Hardware security modules secure primary root encryption key spaces.", "domain": "Security", "risk": "low"},
    {"text": "Distributed denial of service shielding filters malicious edge network traffic.", "domain": "Security", "risk": "medium"},
    {"text": "Role-based authorization interceptors block invalid system API endpoint queries.", "domain": "Security", "risk": "low"},
    {"text": "Zero-trust verification protocols mandate token authentication on every sub-call.", "domain": "Security", "risk": "low"},
    {"text": "End-to-end transport encryption layer configuration isolates data transit pipelines.", "domain": "Security", "risk": "low"},
    {"text": "Phishing detection heuristics scan internal enterprise messaging attachments.", "domain": "Security", "risk": "high"},
    {"text": "Vulnerability scanning cronjobs run source file dependency scanning routines.", "domain": "Security", "risk": "medium"},
    {"text": "Anomalous login checkers alert security operations of unfamiliar geographic sessions.", "domain": "Security", "risk": "medium"},
    {"text": "API key rotating orchestrators deprecate old database access credentials cleanly.", "domain": "Security", "risk": "low"}
]

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Extract sentences from raw_data
sentences = [item['text'] for item in raw_data]

# 1. Initialize an in-memory client
basic_chroma_client = chromadb.EphemeralClient()

# 2. Set up our embedding model function (all-MiniLM-L6-v2)
# Chroma will use this function to automatically turn text into vectors under the hood
hf_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 3. Create a clean collection (think of this as a DB Table named 'fintech_basic')
# Check if collection exists and delete it to ensure a clean state for this example
try:
    basic_chroma_client.delete_collection(name="fintech_basic")
except: # Collection doesn't exist, so no need to delete
    pass
basic_collection = basic_chroma_client.create_collection(
    name="fintech_basic",
    embedding_function=hf_embedding_func
)

# 4. Prepare data for insertion (sentences from our earlier steps)
# Chroma requires a unique ID string for every single document item
basic_ids = [f"id_{i}" for i in range(len(sentences))]

# 5. Insert the sentences into the collection
basic_collection.add(
    documents=sentences,
    ids=basic_ids
)

print(f"Collection created successfully! Total items stored: {basic_collection.count()}")

In [ ]:
# 1. Define your natural language search query
db_query = "What options do consumers have for splitting bills dynamically?"

# 2. Query the collection directly
# Notice we pass raw text! Chroma automatically calls the embedding model,
# converts this query into a 384-dimension vector, and scans its index.
query_results = basic_collection.query(
    query_texts=[db_query],
    n_results=2  # Return top 2 closest matches
)

# 3. Print out the structured results returned from the database
print(f"Search Query: '{db_query}'\n")
print("Top Database Matches:")
print("-" * 40)

for i in range(len(query_results['documents'][0])):
    matched_doc = query_results['documents'][0][i]
    matched_id = query_results['ids'][0][i]
    distance_score = query_results['distances'][0][i]

    print(f"Result #{i+1} [ID: {matched_id}]")
    print(f"Distance Score: {distance_score:.4f}")
    print(f"👉 Text: {matched_doc}\n")

In [ ]:
# Equivalent to: SELECT * FROM fintech_basic LIMIT 5;
sample_records = basic_collection.peek(limit=5)

print("--- PEEKING AT FIRST 5 RECORDS ---")
for i in range(len(sample_records['ids'])):
    print(f"ID: {sample_records['ids'][i]} | Text: {sample_records['documents'][i]}")

In [ ]:
# Equivalent to: SELECT * FROM fintech_basic;
all_records = basic_collection.get()

# Let's inspect what the database returned
print("--- FULL DATABASE DUMP ---")
print(f"Total records retrieved: {len(all_records['ids'])}")
print(f"IDs: {all_records['ids']}")
print(f"Documents: {all_records['documents']}")
print(f"Metadata maps: {all_records['metadatas']}")

In [ ]:
# Explicitly ask Chroma to return the vectors (embeddings) as well
db_rows_with_vectors = basic_collection.get(include=["documents", "metadatas", "embeddings"])

# Let's inspect just the first row's mathematical array
first_vector = db_rows_with_vectors['embeddings'][0]
print(f"First vector contains {len(first_vector)} numbers. Snippet: {first_vector[:5]}...")

In [ ]:
# 1. Initialize an ephemeral (in-memory) ChromaDB Client
chroma_client = chromadb.EphemeralClient()

# 2. Tell Chroma to use our chosen Hugging Face model for automated embedding generation
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 3. Create a collection (equivalent to an enterprise SQL Table or Elasticsearch Index)
collection = chroma_client.create_collection(name="fintech_knowledge_base", embedding_function=hf_ef)

# 4. Prepare data arrays for bulk insertion
documents_list = [item["text"] for item in raw_data]
metadatas_list = [{"domain": item["domain"], "risk": item["risk"]} for item in raw_data]
ids_list = [f"doc_id_{i}" for i in range(len(raw_data))]

# 5. Insert into ChromaDB (It handles string-to-vector embedding generation automatically)
collection.add(
    documents=documents_list,
    metadatas=metadatas_list,
    ids=ids_list
)
print(f"Successfully indexed {collection.count()} items into ChromaDB.\n")

# ==========================================
# 6. EXECUTE HYBRID SEARCH (Vector Similarity + Metadata Metadata Filter)
# ==========================================
search_query = "How do we prevent malicious hacking or system compromises?"

print(f"Executing Hybrid Query: '{search_query}'")
print("Filter: Only show results belonging to the 'Security' domain where risk is 'low'.\n")

query_results = collection.query(
    query_texts=[search_query],
    n_results=3,
    where={
        "$and": [
            {"domain": {"$eq": "Security"}},
            {"risk": {"$eq": "low"}}
        ]
    }
)

# Parse and display the structured database response
for i in range(len(query_results['documents'][0])):
    doc = query_results['documents'][0][i]
    meta = query_results['metadatas'][0][i]
    distance = query_results['distances'][0][i] # Chroma outputs distance (Lower distance = higher similarity)
    print(f"Match {i+1} [Distance Score: {distance:.4f}]")
    print(f"📂 Metadata: {meta}")
    print(f"👉 Text: {doc}\n")

# Task
The user wants to understand Hierarchical Navigable Small Worlds (HNSW) indexing, compare different vector databases (ChromaDB, Pinecone, Qdrant), and learn how to use ChromaDB persistence. The final step will be to summarize the key takeaways and provide further resources.

## Explain HNSW Indexing

### Subtask:
Provide a detailed explanation of the Hierarchical Navigable Small Worlds (HNSW) algorithm.


### Hierarchical Navigable Small Worlds (HNSW) Indexing: Purpose in ANN Search

**Approximate Nearest Neighbor (ANN) Search** is a critical component in applications that deal with high-dimensional data, such as semantic search, recommendation systems, image recognition, and anomaly detection. In these scenarios, exact nearest neighbor search, which involves comparing a query vector to every single vector in a dataset, becomes computationally infeasible as the dataset size grows. This is where ANN search algorithms come into play.

The **purpose of HNSW indexing** is to enable highly efficient and fast ANN search in large-scale vector datasets. HNSW aims to find vectors that are "close enough" to a given query vector, sacrificing a small amount of accuracy for significant gains in search speed and scalability. It achieves this by structuring the high-dimensional vector space in a way that allows for rapid traversal to relevant neighbors, rather than an exhaustive search.

In essence, HNSW provides a practical solution for quickly retrieving similar items from massive datasets, making real-time applications involving vector embeddings feasible.

### How HNSW Works: The Multi-Layer Graph Structure and Search Process

HNSW builds a multi-layer graph structure to enable efficient approximate nearest neighbor (ANN) search. Imagine multiple interconnected graphs, each with a different density of connections, stacked on top of each other.

**1. The Multi-Layer Graph Structure:**
*   **Layers:** HNSW constructs a hierarchy of graphs. The topmost layers contain a sparse set of nodes with long-range connections, acting as

### How HNSW Works: The Multi-Layer Graph Structure and Search Process

HNSW builds a multi-layer graph structure to enable efficient approximate nearest neighbor (ANN) search. Imagine multiple interconnected graphs, each with a different density of connections, stacked on top of each other.

**1. The Multi-Layer Graph Structure:**
*   **Layers:** HNSW constructs a hierarchy of graphs. The topmost layers contain a sparse set of nodes with long-range connections, acting as "expressways" to quickly traverse large distances in the vector space. As you move down to lower layers, the graphs become denser, with more nodes and shorter-range connections. The bottommost layer contains all data points and has very fine-grained connections.
*   **Nodes:** Each node in the graph represents a data point (vector) in your dataset.
*   **Edges (Connections):** Edges connect nodes that are

### How HNSW Works: The Multi-Layer Graph Structure and Search Process

HNSW builds a multi-layer graph structure to enable efficient approximate nearest neighbor (ANN) search. Imagine multiple interconnected graphs, each with a different density of connections, stacked on top of each other.

**1. The Multi-Layer Graph Structure:**
*   **Layers:** HNSW constructs a hierarchy of graphs. The topmost layers contain a sparse set of nodes with long-range connections, acting as "expressways" to quickly traverse large distances in the vector space. As you move down to lower layers, the graphs become denser, with more nodes and shorter-range connections. The bottommost layer contains all data points and has very fine-grained connections.
*   **Nodes:** Each node in the graph represents a data point (vector) in your dataset.
*   **Edges (Connections):** Edges connect nodes that are close to each other in the vector space. The number of connections a node has is a parameter (`M`) that can be tuned. This ensures that each node has a reasonable number of neighbors to explore.

**2. The Search Process:**
*   **Entry Point:** Search usually begins at a random entry point in the topmost layer or a pre-defined entry node.
*   **Greedy Search (Top-Down):** Starting from the entry point, the algorithm performs a greedy search, moving to the neighbor closest to the query vector within the current layer. This process is repeated until it finds a local minimum—a node where no neighbor in the current layer is closer to the query than the current node itself.
*   **Layer Traversal:** Once a local minimum is found in a higher layer, the algorithm

### How HNSW Works: The Multi-Layer Graph Structure and Search Process

HNSW builds a multi-layer graph structure to enable efficient approximate nearest neighbor (ANN) search. Imagine multiple interconnected graphs, each with a different density of connections, stacked on top of each other.

**1. The Multi-Layer Graph Structure:**
*   **Layers:** HNSW constructs a hierarchy of graphs. The topmost layers contain a sparse set of nodes with long-range connections, acting as "expressways" to quickly traverse large distances in the vector space. As you move down to lower layers, the graphs become denser, with more nodes and shorter-range connections. The bottommost layer contains all data points and has very fine-grained connections.
*   **Nodes:** Each node in the graph represents a data point (vector) in your dataset.
*   **Edges (Connections):** Edges connect nodes that are close to each other in the vector space. The number of connections a node has is a parameter (`M`) that can be tuned. This ensures that each node has a reasonable number of neighbors to explore.

**2. The Search Process:**
*   **Entry Point:** Search usually begins at a random entry point in the topmost layer or a pre-defined entry node.
*   **Greedy Search (Top-Down):** Starting from the entry point, the algorithm performs a greedy search, moving to the neighbor closest to the query vector within the current layer. This process is repeated until it finds a local minimum—a node where no neighbor in the current layer is closer to the query than the current node itself.
*   **Layer Traversal:** Once a local minimum is found in a higher layer, the algorithm "drops down" to the corresponding node in the next lower layer. It then repeats the greedy search in this denser layer. This process continues until the lowest layer is reached. In the lowest layer, the search is more exhaustive to find the true approximate nearest neighbors.
*   **Result Set Construction:** Throughout the search, the algorithm maintains a dynamic list of `ef` (exploration factor) closest candidates found so far. This `ef` parameter controls the trade-off between search quality and speed.

**Key Benefits of HNSW:**
*   **Speed:** Extremely fast query times, even for billions of vectors, due to the hierarchical structure that quickly narrows down the search space.
*   **Accuracy:** Achieves high recall (finds most of the true nearest neighbors) with reasonable `ef` and `M` parameter tuning.
*   **Scalability:** Efficiently handles large datasets and can be updated incrementally.
*   **Flexibility:** Works with various distance metrics (e.g., Euclidean, Cosine Similarity).

This combination of multi-layer graphs and greedy search allows HNSW to efficiently navigate high-dimensional spaces, making it a popular choice for modern vector databases and similarity search applications.

## Compare Vector Databases

### Subtask:
Discuss the key trade-offs, features, and use cases of ChromaDB, Pinecone, and Qdrant, highlighting their strengths and weaknesses.


### ChromaDB

ChromaDB is an **open-source, lightweight vector database** designed for simplicity and ease of use, making it particularly appealing for local development, research, and smaller-scale applications.

**Core Features:**
*   **Embeddings Management:** Simplifies the process of creating, storing, and querying vector embeddings.
*   **Metadata Filtering:** Allows for pre- and post-filtering of vector search results based on associated metadata, enabling hybrid search capabilities.
*   **Integrated Embedding Functions:** Often comes with built-in or easy-to-integrate embedding models, abstracting away some of the complexities of embedding generation.
*   **Persistence Options:** Supports both in-memory (ephemeral) and disk-based (persistent) storage, offering flexibility depending on the use case.

**Advantages:**
*   **Open-Source & Local-First:** Being open-source means it's free to use, highly customizable, and can be run entirely locally without requiring external cloud services. This is great for privacy-sensitive applications or environments with limited internet access.
*   **Ease of Use:** Its API is designed to be straightforward, making it quick to get started for developers new to vector databases.
*   **Embedded Mode:** Can run as an embedded database within your application, reducing deployment overhead.
*   **Cost-Effective:** No infrastructure costs for local deployments.
*   **Good for Prototyping:** Excellent for quickly building and testing RAG (Retrieval Augmented Generation) applications and other vector search prototypes.

**Limitations & Use Cases:**
*   **Scalability:** While improving, its horizontal scalability for massive, petabyte-scale datasets and extremely high query loads is generally not as robust or mature as enterprise-grade, cloud-native solutions like Pinecone.
*   **Performance:** For extremely low-latency requirements on very large datasets, dedicated distributed systems might offer better performance.
*   **Managed Services:** Currently, a fully managed cloud offering from Chroma is still evolving, meaning users often manage their own deployments for larger needs.
*   **Best Suited For:** Local development, small to medium-scale applications, proof-of-concepts, educational purposes, and scenarios where data locality and cost efficiency are primary concerns.

### Pinecone

Pinecone is a **fully managed, cloud-native vector database** service built for large-scale, high-performance similarity search. It focuses on providing a production-ready, scalable infrastructure for vector embeddings, abstracting away the complexities of managing distributed systems.

**Core Features:**
*   **Fully Managed Service:** Handles all infrastructure, scaling, and maintenance automatically.
*   **High Performance & Scalability:** Optimized for low-latency queries and can scale to billions of vectors.
*   **Real-time Indexing:** Supports real-time updates and deletions of vectors.
*   **Metadata Filtering:** Offers powerful filtering capabilities based on metadata for precise search.
*   **Hybrid Search:** Combines vector similarity with keyword search for enhanced relevance.
*   **Multi-tenancy:** Allows for isolated environments for different applications or teams.

**Advantages:**
*   **Zero Infrastructure Management:** Users don't need to worry about servers, scaling, or database operations.
*   **Enterprise-Grade Scalability:** Designed from the ground up to handle massive datasets and high query throughput in production environments.
*   **Optimized Performance:** Provides consistently low latency even under heavy loads.
*   **Rich Feature Set:** Advanced filtering, hybrid search, and security features make it suitable for complex applications.
*   **Reliability & Uptime:** As a managed service, it offers high availability and redundancy.

**Limitations & Use Cases:**
*   **Cost:** Being a managed cloud service, it can be more expensive, especially for smaller projects or those with fluctuating usage.
*   **Vendor Lock-in:** Tightly coupled to the Pinecone ecosystem, which might limit flexibility compared to open-source alternatives.
*   **Less Control:** Users have less granular control over the underlying infrastructure and optimization parameters compared to self-hosted solutions.
*   **No Local Mode:** Primarily cloud-based, so it's not suitable for purely local development or applications requiring strict data residency without cloud reliance.
*   **Best Suited For:** Production-grade applications, large-scale semantic search, recommendation engines, generative AI applications, and enterprises requiring high performance, scalability, and minimal operational overhead.

### Qdrant

Qdrant is an **open-source, production-ready vector similarity search engine** that can be deployed on your infrastructure or used as a managed cloud service. It's designed for high-performance and scalability, offering a rich set of features for complex vector search scenarios.

**Core Features:**
*   **Filtering Capabilities:** Extremely powerful filtering capabilities, including structured filtering, geospatial search, and payload filtering, allowing for highly specific and combined searches.
*   **Quantization:** Supports various quantization methods (e.g., product quantization, scalar quantization) to reduce memory footprint and improve query speed, especially for very large datasets.
*   **Hybrid Cloud & On-Premise:** Can be self-hosted on various cloud providers or on-premise, and also offers a managed cloud service.
*   **Scalability & High Availability:** Designed for distributed deployment, ensuring high availability and horizontal scalability.
*   **Collections & Snapshots:** Organizes vectors into named collections and supports snapshotting for data backup and recovery.
*   **Multiple Distance Metrics:** Supports various distance metrics (cosine, Euclidean, dot product).

**Advantages:**
*   **Open-Source & Flexible Deployment:** Being open-source allows for full control and customization, and its flexibility in deployment (self-hosted or managed) caters to diverse operational needs.
*   **Advanced Filtering:** Qdrant's filtering is often cited as one of its strongest features, enabling precise pre-filtering based on metadata before vector similarity search.
*   **Performance Optimization:** Features like quantization provide significant performance boosts and memory savings, crucial for very large-scale applications.
*   **Production-Ready:** Built with production needs in mind, offering features like high availability, replication, and data consistency.
*   **Active Community & Ecosystem:** Benefits from an active open-source community and integrates well with the broader AI/ML ecosystem.

**Limitations & Use Cases:**
*   **Complexity:** While powerful, its extensive features and configuration options can introduce a steeper learning curve compared to more minimalist solutions like ChromaDB.
*   **Resource Management:** For self-hosted deployments, users are responsible for managing infrastructure, scaling, and maintenance, which requires operational expertise.
*   **Managed Service Cost:** The managed cloud service comes with associated costs, similar to other cloud-native solutions.
*   **Best Suited For:** Complex search scenarios requiring advanced filtering, large-scale production deployments (self-hosted or managed), developers who need granular control over their vector index, and applications where performance and memory efficiency are paramount.

## Demonstrate ChromaDB Persistence

### Subtask:
Show how to create and use a persistent ChromaDB client so that data is saved to disk and can be reloaded in future sessions.


**Reasoning**:
Initialize a persistent ChromaDB client, specifying a path for data storage, and ensure a clean state by removing any existing data at that path.



In [ ]:
import chromadb
import shutil
import os

# Define the persistence path
CHROMA_PATH = "/tmp/chromadb_data"

# Ensure a clean state for the demonstration
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print(f"Removed existing ChromaDB data at: {CHROMA_PATH}")

# 1. Create a persistent ChromaDB client
persistent_client = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"Persistent ChromaDB client initialized at: {CHROMA_PATH}")

**Reasoning**:
Now that the persistent client is initialized, I will create a collection, add documents with metadata to it, and confirm the data has been stored. This demonstrates the first part of persistence: writing data.



In [ ]:
from chromadb.utils import embedding_functions

# 2. Set up our embedding model function (all-MiniLM-L6-v2)
hf_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 3. Create a collection using the persistent client
# Check if collection exists and delete it to ensure a clean state for this example
try:
    persistent_client.delete_collection(name="fintech_persistent")
except:
    pass

persistent_collection = persistent_client.create_collection(
    name="fintech_persistent",
    embedding_function=hf_embedding_func
)

# 4. Prepare data arrays for bulk insertion (reusing raw_data from previous steps)
documents_list = [item["text"] for item in raw_data]
metadatas_list = [{"domain": item["domain"], "risk": item["risk"]} for item in raw_data]
ids_list = [f"pers_id_{i}" for i in range(len(raw_data))]

# 5. Insert into ChromaDB
persistent_collection.add(
    documents=documents_list,
    metadatas=metadatas_list,
    ids=ids_list
)

print(f"Successfully indexed {persistent_collection.count()} items into the persistent ChromaDB collection 'fintech_persistent'.")

**Reasoning**:
Now that data has been added to the persistent collection, I will simulate a new session by re-initializing the ChromaDB client and loading the existing collection. Then, I will query this loaded collection to demonstrate that the data has persisted on disk.



In [ ]:
print("\n--- Simulating a new session by re-initializing ChromaDB client ---")

# 6. Re-initialize the persistent client (simulating a new session)
# The CHROMA_PATH already exists and contains our data
reloaded_client = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"Reloaded ChromaDB client initialized from: {CHROMA_PATH}")

# 7. Get the previously created collection
reloaded_collection = reloaded_client.get_collection(
    name="fintech_persistent",
    embedding_function=hf_embedding_func # Must pass the same embedding function
)

print(f"Successfully loaded collection 'fintech_persistent' with {reloaded_collection.count()} items.")

# 8. Perform a query on the reloaded collection to demonstrate persistence
search_query = "How can I manage credit risk and loan approvals?"
print(f"\nExecuting query on reloaded collection: '{search_query}'")

query_results_reloaded = reloaded_collection.query(
    query_texts=[search_query],
    n_results=2,
    where={
        "domain": {"$eq": "Lending"} # Filter for Lending domain
    }
)

print("\nTop Database Matches from Reloaded Collection (Filtered by 'Lending' domain):")
print("-" * 70)

for i in range(len(query_results_reloaded['documents'][0])):
    doc = query_results_reloaded['documents'][0][i]
    meta = query_results_reloaded['metadatas'][0][i]
    distance = query_results_reloaded['distances'][0][i]
    print(f"Result #{i+1} [Distance Score: {distance:.4f}]")
    print(f"📂 Metadata: {meta}")
    print(f"👉 Text: {doc}\n")

## Final Task

### Subtask:
Summarize the key takeaways from the explanations and demonstrations, and provide resources for further learning on embeddings and vector databases.


## Final Task

### Subtask:
Summarize the key takeaways from the explanations and demonstrations, and provide resources for further learning on embeddings and vector databases.


In [ ]:
# Install pypdf and chromadb for reading PDF documents and vector database operations
!pip install -q pypdf chromadb

import pypdf
import os
from chromadb.utils import embedding_functions
import chromadb
import shutil

# Define the path to your PDF document
PDF_PATH = "/content/test_embed.pdf"

# Define the persistence path for ChromaDB
CHROMA_PERSIST_PATH_PDF = "./chromadb_rbi_data"

print(f"Expecting PDF at: {PDF_PATH}")
print(f"ChromaDB persistence path: {CHROMA_PERSIST_PATH_PDF}")

### 2. Load and Chunk the PDF Content

We will now load the PDF document and extract its text content. Since large documents are not suitable for direct embedding, we'll split the text into smaller, overlapping chunks. This helps maintain context within each chunk while ensuring they are small enough for effective embedding.

In [ ]:
# Function to load a PDF and extract text
def load_pdf(file_path):
    text = ""
    try:
        with open(file_path, 'rb') as f:
            reader = pypdf.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() + "\n"
    except FileNotFoundError:
        print(f"Error: PDF file not found at {file_path}. Please ensure it's uploaded.")
        return None
    return text

# Function to chunk text
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    chunks = []
    words = text.split()
    # Simple word-based chunking
    # More sophisticated chunking might involve sentence tokenization or recursive splitting
    for i in range(0, len(words), chunk_size - chunk_overlap):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Load the PDF content
rbi_text = load_pdf(PDF_PATH)

if rbi_text:
    # Chunk the text
    rbi_chunks = chunk_text(rbi_text)
    print(f"Loaded PDF and created {len(rbi_chunks)} chunks.")
    # Display first few chunks
    for i, chunk in enumerate(rbi_chunks[:3]):
        print(f"--- Chunk {i+1} ---")
        print(chunk[:200], "...") # Print first 200 chars of chunk
else:
    print("Could not process PDF. Please check the file path and ensure the PDF is uploaded.")
    rbi_chunks = []

In [ ]:
# Ensure a clean state for the ChromaDB persistence path
# Define the persistence path for ChromaDB in /tmp for better write permissions
CHROMA_PERSIST_PATH_PDF = "/tmp/chromadb_rbi_data_pdf"

if os.path.exists(CHROMA_PERSIST_PATH_PDF):
    try:
        shutil.rmtree(CHROMA_PERSIST_PATH_PDF)
        print(f"Removed existing ChromaDB data at: {CHROMA_PERSIST_PATH_PDF}")
    except Exception as e:
        print(f"WARNING: Could not remove existing ChromaDB data directory at {CHROMA_PERSIST_PATH_PDF}: {e}")
        print("This might lead to a read-only database issue or stale data. Attempting to proceed.")

# Initialize a persistent ChromaDB client
pdf_chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_PATH_PDF)
print(f"Persistent ChromaDB client initialized at: {CHROMA_PERSIST_PATH_PDF}")

# Set up the embedding model function
# We'll use the same all-MiniLM-L6-v2 model for consistency
hf_embedding_func_pdf = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create a new collection for the RBI document, ensuring a clean state by deleting if it exists
collection_name = "rbi_documents"
try:
    # Attempt to delete the collection first to ensure a clean state for re-indexing
    pdf_chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing ChromaDB collection: {collection_name}")
except:
    # If deletion fails (e.g., collection doesn't exist), proceed without error
    pass

rbi_collection = pdf_chroma_client.create_collection(
    name=collection_name,
    embedding_function=hf_embedding_func_pdf
)

if rbi_chunks:
    # Prepare data for insertion
    rbi_chunk_ids = [f"doc_{i}" for i in range(len(rbi_chunks))]
    rbi_chunk_metadatas = [{
        "source": PDF_PATH,
        "chunk_id": i,
        "length": len(chunk)
    } for i, chunk in enumerate(rbi_chunks)]

    # Add chunks to the ChromaDB collection
    rbi_collection.add(
        documents=rbi_chunks,
        metadatas=rbi_chunk_metadatas,
        ids=rbi_chunk_ids
    )

    print(f"Successfully indexed {rbi_collection.count()} chunks from '{PDF_PATH}' into ChromaDB.")
else:
    print("No chunks to index. Please ensure PDF loading and chunking were successful.")

### 4. Create a Query Function

Now that the PDF chunks are embedded and stored in ChromaDB, we can create a simple query function. This function will take a natural language question, embed it, and then query the ChromaDB collection to find the most relevant document chunks based on semantic similarity.

In [ ]:
def query_chromadb(query_text, collection, n_results=3, where=None):
    """
    Queries the ChromaDB collection for the most relevant documents, with optional metadata filtering.

    Args:
        query_text (str): The natural language query string.
        collection: The ChromaDB collection object to query.
        n_results (int): The number of top results to retrieve.
        where (dict, optional): A dictionary for metadata filtering. Defaults to None.

    Returns:
        list: A list of dictionaries, each containing 'document', 'metadata', and 'distance'.
    """
    query_results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances'],
        where=where
    )

    formatted_results = []
    if query_results and query_results['documents'] and query_results['documents'][0]:
        for i in range(len(query_results['documents'][0])):
            formatted_results.append({
                'document': query_results['documents'][0][i],
                'metadata': query_results['metadatas'][0][i],
                'distance': query_results['distances'][0][i]
            })
    return formatted_results

print("Query function 'query_chromadb' defined and updated with 'where' clause support.")

### 5. Ask Example Questions

Let's use our `query_chromadb` function to ask a few questions and see how well it retrieves relevant information from our indexed PDF content.

### Structuring Complex Filter Dictionaries for ChromaDB's `where` Clause

The `where` clause in ChromaDB's `query()` method accepts a dictionary that defines your filtering criteria. This dictionary can be simple, containing a single key-value pair for an exact match, or complex, involving nested logical and comparison operators.

**Basic Structure for a Single Condition:**

For a simple equality check (the default if no operator is specified, or explicitly with `$eq`):

```python
{"metadata_key": "value"} # Implicit $eq
# or
{"metadata_key": {"$eq": "value"}}
```

For other comparison operators:

```python
{"metadata_key": {"$gt": 100}} # Greater than
{"metadata_key": {"$ne": "exclude_value"}} # Not equal to
{"metadata_key": {"$in": ["value1", "value2"]}} # Value is in a list
```

**Combining Conditions with Logical Operators (`$and`, `$or`):**

When you need to apply multiple conditions, you use the `$and` or `$or` operators. These operators take a list of filter dictionaries as their value.

*   **`$and` Operator:** All conditions within the `$and` list must be true for a document to be returned.

    ```python
    {
        "$and": [
            {"metadata_key1": {"$eq": "value1"}},  # Condition 1
            {"metadata_key2": {"$gt": 50}},       # Condition 2
            {"metadata_key3": {"$in": ["A", "B"]}}  # Condition 3
        ]
    }
    ```

*   **`$or` Operator:** At least one of the conditions within the `$or` list must be true for a document to be returned.

    ```python
    {
        "$or": [
            {"metadata_key1": {"$eq": "value_a"}}, # Condition 1
            {"metadata_key1": {"$eq": "value_b"}}  # Condition 2
        ]
    }
    ```

**Nesting Logical Operators (Advanced):**

You can also nest `$and` and `$or` operators to build even more complex logic. For example, to find documents that satisfy `(Condition1 AND Condition2) OR Condition3`:

```python
{
    "$or": [
        {
            "$and": [
                {"metadata_key_A": {"$eq": "specific"}},
                {"metadata_key_B": {"$gt": 1000}}
            ]
        },
        {"metadata_key_C": {"$in": ["typeX", "typeY"]}}
    ]
}
```

**Common Operators:**

*   `$eq`: Equal to (e.g., `{"category": {"$eq": "finance"}}`)
*   `$ne`: Not equal to (e.g., `{"status": {"$ne": "draft"}}`)
*   `$gt`: Greater than (e.g., `{"views": {"$gt": 1000}}`)
*   `$gte`: Greater than or equal to (e.g., `{"score": {"$gte": 0.8}}`)
*   `$lt`: Less than (e.g., `{"age": {"$lt": 30}}`)
*   `$lte`: Less than or equal to (e.g., `{"price": {"$lte": 50}}`)
*   `$in`: Value is in a list (e.g., `{"tags": {"$in": ["AI", "ML"]}}`)
*   `$nin`: Value is not in a list (e.g., `{"region": {"$nin": ["EU", "ASIA"]}}`)

By understanding these structures and operators, you can precisely control which documents are considered during your vector similarity searches, significantly enhancing the relevance and accuracy of your retrieval augmented generation (RAG) applications.

In [ ]:
print("\n--- Demonstrating a MORE Complex Filter Dictionary ---")

complex_filtered_question = "What are the regulatory guidelines and policies for financial innovation?"

# Example: Combine conditions to find chunks where:
# (length > 3500 AND chunk_id == 13) OR (chunk_id == 32 AND source == '/content/test_embed.pdf')
complex_filter = {
    "$or": [
        {
            "$and": [
                {"length": {"$gt": 3500}},
                {"chunk_id": {"$eq": 13}}
            ]
        },
        {
            "$and": [
                {"chunk_id": {"$eq": 32}},
                {"source": {"$eq": "/content/test_embed.pdf"}}
            ]
        }
    ]
}

print(f"Query: '{complex_filtered_question}' (complex filter with $or and $and combinations)")
complex_results = query_chromadb(
    complex_filtered_question,
    rbi_collection,
    n_results=3,
    where=complex_filter
)

if complex_results:
    for j, res in enumerate(complex_results):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}] - Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:200]}...")
else:
    print("No results found matching the complex filter criteria.")

In [ ]:
example_questions = [
    "What are the guidelines for monetary policy?",
    "What is the role of the Reserve Bank of India in financial stability?",
    "Can you explain regulations on digital payments?",
    "What measures are taken for inflation control?",
    "How does RBI manage foreign exchange reserves?"
]

for i, question in enumerate(example_questions):
    print(f"\n--- Question {i+1}: {question} ---")
    results = query_chromadb(question, rbi_collection, n_results=2)

    if results:
        for j, res in enumerate(results):
            print(f"Match {j+1} [Distance: {res['distance']:.4f}]")
            print(f"Metadata: {res['metadata']}")
            print(f"Document: {res['document'][:500]}...") # Print first 500 characters of the document
    else:
        print("No relevant results found.")

### 5.1 Demonstrate Query with Metadata Filtering

Now, let's see how we can use the `where` clause to filter our search results based on metadata. For example, we might want to find information from a specific chunk, or chunks that meet certain length criteria. Here, we'll demonstrate filtering for chunks with a specific `chunk_id`.

In [ ]:
print("\n--- Demonstrating Query with Metadata Filtering ---")
filtered_question = "What are the guidelines for monetary policy?"

# Example: Filter results to include only chunks with 'chunk_id' equal to 32
# (based on a relevant chunk from previous example questions)
filtered_results = query_chromadb(
    filtered_question,
    rbi_collection,
    n_results=1,
    where={"chunk_id": {"$eq": 32}}
)

print(f"Query: '{filtered_question}' (filtered for chunk_id = 32)")
if filtered_results:
    for j, res in enumerate(filtered_results):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}]")
        print(f"Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:500]}...")
else:
    print("No results found matching the query and filter criteria.")

# Another example: Filter results to include only chunks with 'length' greater than 3500
print("\n--- Another Filter Example: Chunks with length > 3500 ---")
filtered_results_by_length = query_chromadb(
    filtered_question,
    rbi_collection,
    n_results=2,
    where={"length": {"$gt": 3500}}
)

print(f"Query: '{filtered_question}' (filtered for length > 3500)")
if filtered_results_by_length:
    for j, res in enumerate(filtered_results_by_length):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}]")
        print(f"Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:500]}...")
else:
    print("No results found matching the query and filter criteria.")

### 5.2 Advanced Metadata Filtering with ChromaDB

ChromaDB's `where` clause in the `.query()` method allows for powerful and flexible metadata filtering. This enables you to combine semantic similarity search with structured filtering, significantly improving the relevance of your results. You can use various operators to create complex conditions.

**Key Operators:**
*   `$eq`: Equal to
*   `$ne`: Not equal to
*   `$gt`: Greater than
*   `$gte`: Greater than or equal to
*   `$lt`: Less than
*   `$lte`: Less than or equal to
*   `$in`: Value is in a list
*   `$nin`: Value is not in a list
*   `$and`: Logical AND for combining multiple conditions
*   `$or`: Logical OR for combining multiple conditions

These operators allow you to narrow down your search space before or after the vector similarity search, ensuring that only documents meeting your specific criteria are considered.

In [ ]:
# Define a common query for demonstration
common_query = "What are the regulatory frameworks for financial technology?"

print(f"\n--- Demonstrating Advanced Query with Metadata Filtering ---\n")

# Example 1: Combining conditions with $and - Filter by source AND length greater than a value
print("Query: 'What are the regulatory frameworks for financial technology?' (filtered by source AND length > 3000)")
results_and_condition = query_chromadb(
    common_query,
    rbi_collection,
    n_results=2,
    where={
        "$and": [
            {"source": {"$eq": "/content/test_embed.pdf"}},
            {"length": {"$gt": 3000}}
        ]
    }
)

if results_and_condition:
    for j, res in enumerate(results_and_condition):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}] - Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:200]}...")
else:
    print("No results found matching the query and AND filter criteria.")

print("\n" + "-"*70 + "\n")

# Example 2: Using $or - Filter by specific chunk_ids
print("Query: 'What are the regulatory frameworks for financial technology?' (filtered by chunk_id IN [13, 32])")
results_or_condition = query_chromadb(
    common_query,
    rbi_collection,
    n_results=2,
    where={
        "$or": [
            {"chunk_id": {"$eq": 13}},
            {"chunk_id": {"$eq": 32}}
        ]
    }
)

if results_or_condition:
    for j, res in enumerate(results_or_condition):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}] - Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:200]}...")
else:
    print("No results found matching the query and OR filter criteria.")

print("\n" + "-"*70 + "\n")

# Example 3: Using $ne (not equal to) for chunk_id
print("Query: 'What are the regulatory frameworks for financial technology?' (filtered by chunk_id NOT equal to 50)")
results_ne_condition = query_chromadb(
    common_query,
    rbi_collection,
    n_results=2,
    where={
        "chunk_id": {"$ne": 50}
    }
)

if results_ne_condition:
    for j, res in enumerate(results_ne_condition):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}] - Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:200]}...")
else:
    print("No results found matching the query and NOT EQUAL filter criteria.")

print("\n" + "-"*70 + "\n")

# Example 4: Using $in (value in a list) for chunk_id
print("Query: 'What are the regulatory frameworks for financial technology?' (filtered by chunk_id IN [0, 1, 2])")
results_in_condition = query_chromadb(
    common_query,
    rbi_collection,
    n_results=2,
    where={
        "chunk_id": {"$in": [0, 1, 2]}
    }
)

if results_in_condition:
    for j, res in enumerate(results_in_condition):
        print(f"Match {j+1} [Distance: {res['distance']:.4f}] - Metadata: {res['metadata']}")
        print(f"Document: {res['document'][:200]}...")
else:
    print("No results found matching the query and IN filter criteria.")

### 6. Push the Notebook to GitHub

To save your work and share this notebook, you can push it to a GitHub repository. Follow these steps:

1.  **Save your notebook:** Ensure all your changes are saved in Colab (`File > Save`).
2.  **Open GitHub integration:** Go to `File > Save a copy in GitHub`.
3.  **Authenticate (if necessary):** If you haven't connected Colab to GitHub before, you will be prompted to authenticate.
4.  **Select Repository:** Choose the GitHub repository where you want to save the notebook. You can also create a new repository directly from this interface.
5.  **Provide a commit message:** Write a brief message describing your changes (e.g., "Initial commit: PDF processing and ChromaDB query function").
6.  **Include a link to Colab (optional):** You can check the option to include a link back to Colab.
7.  **Click OK:** This will push your notebook to the selected GitHub repository.